## <u>Load packages</u>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from simulator_numba import simulate_numba
from simulator import simulate
import pandas as pd
from scipy.stats import norm
import matplotlib.gridspec as gridspec

## <u>Load the data</u>

In [ ]:
# 3 files of observed data, from 40 replications of the true model (with the unknown true parameters)
df_inf = pd.read_csv('data/infected_timeseries.csv') 
df_rew = pd.read_csv('data/rewiring_timeseries.csv')
df_deg = pd.read_csv('data/final_degree_histograms.csv')

# Extracting the observed summary statistics (mean over replications)
obs_inf = df_inf.groupby('time')['infected_fraction'].mean().values 
obs_rewire = df_rew.groupby('time')['rewire_count'].mean().values
obs_deg = df_deg.groupby('degree')['count'].mean().values

## <u>ABC</u>

### Distance function
We measure to what mesure the generated data summary statistic is close to the observed data summary statistics

In [ ]:
def summarize(sim_results):
    sim_inf, sim_rewire, sim_deg = sim_results
    
    return np.array([
        np.max(sim_inf),                          
        np.argmax(sim_inf) / len(sim_inf),        
        np.mean(sim_inf),                         
        sim_inf[-1],                              
        
        np.sum(sim_rewire) / len(sim_rewire),     
        np.argmax(sim_rewire) / len(sim_rewire),  
        
        np.mean(sim_deg),                         
        np.std(sim_deg),
    ])

def calculate_distance(sim_results, obs_summary):
    sim_summary = summarize(sim_results)
    diff = (sim_summary - obs_summary) / (np.abs(obs_summary) + 1e-9)
    return np.mean(diff**2)

def distance_1(sim_results, obs_inf, obs_deg):
    sim_inf, _, sim_deg = sim_results

    d_inf = np.mean((sim_inf - obs_inf)**2)
    d_deg = np.mean((sim_deg - obs_deg)**2) / (np.mean(obs_deg)**2 + 1e-9)

    return d_inf + d_deg

def distance_2(sim_results, obs_inf, obs_deg, obs_rewire):
    sim_inf, sim_rewire, sim_deg = sim_results

    d_inf    = np.mean((sim_inf - obs_inf)**2)
    d_deg    = np.mean((sim_deg - obs_deg)**2) / (np.mean(obs_deg)**2 + 1e-9)
    d_rewire = np.mean((sim_rewire - obs_rewire)**2) / (np.mean(obs_rewire)**2 + 1e-9)

    return d_inf + d_deg + d_rewire

obs_summary = summarize((obs_inf, obs_rewire, obs_deg))

## <u>Rejection ABC</u>

### 1- scalars (8)

In [ ]:
# -----------------------------------------------------------
# ABC rejection sampling
# -----------------------------------------------------------
n_iterations = 100000
accepted_params = []
all_distances = []
all_params = []

for i in range(n_iterations):
    p_beta = np.random.uniform(0.05, 0.5)
    p_gamma = np.random.uniform(0.02, 0.2)
    p_rho = np.random.uniform(0.0, 0.8)
    
    results = simulate_numba(beta=p_beta, gamma=p_gamma, rho=p_rho, N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    
    d = calculate_distance(results, obs_summary)
    # d = distance_1(results, obs_inf, obs_deg)
    # d = distance_2(results, obs_inf, obs_deg, obs_rewire)

    all_distances.append(d)
    all_params.append([p_beta, p_gamma, p_rho])


# -----------------------------------------------------------
# We keep the parameters that produce a distance in the lowest 1% (epsilon)
# -----------------------------------------------------------
epsilon = np.percentile(all_distances, 1)

accepted_params = [
    [all_params[i][0], all_params[i][1], all_params[i][2], all_distances[i]] 
    for i in range(n_iterations) 
    if all_distances[i] <= epsilon
]


# -----------------------------------------------------------
# Results
# -----------------------------------------------------------
res = np.array(accepted_params)

print(f"Estimated means {len(res)}:")
print(f"Beta: {res[:,0].mean():.4f}, Gamma: {res[:,1].mean():.4f}, Rho: {res[:,2].mean():.4f}")


# -----------------------------------------------------------
# Posterior Predictive Check
# -----------------------------------------------------------
ppc_distances = []
for params in res[:, :3]:
    results = simulate_numba(beta=params[0], gamma=params[1], rho=params[2],
                             N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    d = calculate_distance(results, obs_summary)
    ppc_distances.append(d)

print(f"Mean distance PPC: {np.mean(ppc_distances):.6f}")
print(f"Median distance PPC: {np.median(ppc_distances):.6f}")
res_v1 = res

### 2- inf + deg

In [ ]:
# -----------------------------------------------------------
# ABC rejection sampling
# -----------------------------------------------------------
n_iterations = 100000
accepted_params = []
all_distances = []
all_params = []

for i in range(n_iterations):
    p_beta = np.random.uniform(0.05, 0.5)
    p_gamma = np.random.uniform(0.02, 0.2)
    p_rho = np.random.uniform(0.0, 0.8)
    
    results = simulate_numba(beta=p_beta, gamma=p_gamma, rho=p_rho, N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    
    # d = calculate_distance(results, obs_summary)
    d = distance_1(results, obs_inf, obs_deg)
    # d = distance_2(results, obs_inf, obs_deg, obs_rewire)

    all_distances.append(d)
    all_params.append([p_beta, p_gamma, p_rho])


# -----------------------------------------------------------
# We keep the parameters that produce a distance in the lowest 1% (epsilon)
# -----------------------------------------------------------
epsilon = np.percentile(all_distances, 1)

accepted_params = [
    [all_params[i][0], all_params[i][1], all_params[i][2], all_distances[i]] 
    for i in range(n_iterations) 
    if all_distances[i] <= epsilon
]


# -----------------------------------------------------------
# results
# -----------------------------------------------------------
res = np.array(accepted_params)

print(f"Estimated means {len(res)}:")
print(f"Beta: {res[:,0].mean():.4f}, Gamma: {res[:,1].mean():.4f}, Rho: {res[:,2].mean():.4f}")


# -----------------------------------------------------------
# Posterior Predictive Check 
# -----------------------------------------------------------
ppc_distances = []
for params in res[:, :3]:
    results = simulate_numba(beta=params[0], gamma=params[1], rho=params[2],
                             N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    d = distance_1(results, obs_inf, obs_deg)
    ppc_distances.append(d)

print(f"Mean distance PPC: {np.mean(ppc_distances):.6f}")
print(f"Median distance PPC: {np.median(ppc_distances):.6f}")
res_v2 = res

### 3- inf + deg + rewire

In [ ]:
# -----------------------------------------------------------
# ABC rejection sampling
# -----------------------------------------------------------
n_iterations = 100000
accepted_params = []
all_distances = []
all_params = []

for i in range(n_iterations):
    p_beta = np.random.uniform(0.05, 0.5)
    p_gamma = np.random.uniform(0.02, 0.2)
    p_rho = np.random.uniform(0.0, 0.8)
    
    results = simulate_numba(beta=p_beta, gamma=p_gamma, rho=p_rho, N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    
    # d = calculate_distance(results, obs_summary)
    # d = distance_1(results, obs_inf, obs_deg)
    d = distance_2(results, obs_inf, obs_deg, obs_rewire)

    all_distances.append(d)
    all_params.append([p_beta, p_gamma, p_rho])


# -----------------------------------------------------------
# We keep the parameters that produce a distance in the lowest 1% (epsilon)
# -----------------------------------------------------------
epsilon = np.percentile(all_distances, 1)

accepted_params = [
    [all_params[i][0], all_params[i][1], all_params[i][2], all_distances[i]] 
    for i in range(n_iterations) 
    if all_distances[i] <= epsilon
]


# -----------------------------------------------------------
# results
# -----------------------------------------------------------
res = np.array(accepted_params)

print(f"Estimated means {len(res)}:")
print(f"Beta: {res[:,0].mean():.4f}, Gamma: {res[:,1].mean():.4f}, Rho: {res[:,2].mean():.4f}")


# -----------------------------------------------------------
# Posterior Predictive Check 
# -----------------------------------------------------------
ppc_distances = []
for params in res[:, :3]:
    results = simulate_numba(beta=params[0], gamma=params[1], rho=params[2],
                             N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
    d = distance_2(results, obs_inf, obs_deg, obs_rewire)
    ppc_distances.append(d)

print(f"Mean distance PPC: {np.mean(ppc_distances):.6f}")
print(f"Median distance PPC: {np.median(ppc_distances):.6f}")
res_v3 = res

## <u>SMC-ABC</u>

In [ ]:
def smc_abc(n_particles, epsilons, obs_summary, obs_inf):
    
    print(f"Step 0 : initialisation from the prior ({n_particles} particles)...")
    particles = []
    weights   = []
    history   = []
    
    while len(particles) < n_particles:
        p_beta  = np.random.uniform(0.05, 0.5)
        p_gamma = np.random.uniform(0.02, 0.2)
        p_rho   = np.random.uniform(0.0,  0.8)
        
        results = simulate_numba(beta=p_beta, gamma=p_gamma, rho=p_rho,
                                 N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
        d = calculate_distance(results, obs_summary)
        
        if d <= epsilons[0]:
            particles.append([p_beta, p_gamma, p_rho])
            weights.append(1.0)
    
    particles = np.array(particles)
    weights   = np.array(weights)
    weights  /= weights.sum()

    history.append((epsilons[0], particles.copy(), weights.copy()))
    print(f"  → {len(particles)} accepted particles under ε={epsilons[0]}")
    
    for t, epsilon in enumerate(epsilons[1:], start=1):
        print(f"Step {t} : ε={epsilon}...")
        
        cov    = np.cov(particles.T, aweights=weights)
        kernel = 2 * cov 
        
        new_particles = []
        new_weights   = []
        n_tries       = 0
        
        while len(new_particles) < n_particles:
            n_tries += 1
            
            idx = np.random.choice(len(particles), p=weights)
            p   = particles[idx]
            
            candidate = np.random.multivariate_normal(p, kernel)
            p_beta, p_gamma, p_rho = candidate
            
            if not (0.05 <= p_beta  <= 0.5  and
                    0.02 <= p_gamma <= 0.2   and
                    0.0  <= p_rho   <= 0.8):
                continue
            
            results = simulate_numba(beta=p_beta, gamma=p_gamma, rho=p_rho,
                                     N=200, p_edge=0.05, n_infected0=5, T=len(obs_inf)-1)
            d = calculate_distance(results, obs_summary)
            
            if d <= epsilon:
                prior = 1.0  
                
                kernel_sum = np.sum(
                    weights * norm.pdf(p_beta,  particles[:, 0], np.sqrt(kernel[0,0])) *
                                norm.pdf(p_gamma, particles[:, 1], np.sqrt(kernel[1,1])) *
                                norm.pdf(p_rho,   particles[:, 2], np.sqrt(kernel[2,2]))
                )

                w = prior / (kernel_sum + 1e-300)
                new_particles.append([p_beta, p_gamma, p_rho])
                new_weights.append(w)
        
        particles = np.array(new_particles)
        weights   = np.array(new_weights)
        weights  /= weights.sum()
        
        history.append((epsilon, particles.copy(), weights.copy()))

        acc_rate = n_particles / n_tries
        print(f"  → {len(particles)} particules, taux d'acceptation: {acc_rate:.3f}")
        
        # Résultats intermédiaires
        print(f"  Beta:  {np.average(particles[:,0], weights=weights):.4f} "
              f"± {np.sqrt(np.average((particles[:,0] - np.average(particles[:,0], weights=weights))**2, weights=weights)):.4f}")
        print(f"  Gamma: {np.average(particles[:,1], weights=weights):.4f} "
              f"± {np.sqrt(np.average((particles[:,1] - np.average(particles[:,1], weights=weights))**2, weights=weights)):.4f}")
        print(f"  Rho:   {np.average(particles[:,2], weights=weights):.4f} "
              f"± {np.sqrt(np.average((particles[:,2] - np.average(particles[:,2], weights=weights))**2, weights=weights)):.4f}")
    
    return particles, weights, history


# --- SMC-ABC ---
epsilons = [0.5, 0.3, 0.15, 0.08, 0.04, 0.02]

particles, weights, history= smc_abc(
    n_particles = 1000,
    epsilons    = epsilons,
    obs_summary = obs_summary,
    obs_inf     = obs_inf
)

# --- Results ---
print("\n=== Final Results ===")
print(f"Beta:  {np.average(particles[:,0], weights=weights):.4f}")
print(f"Gamma: {np.average(particles[:,1], weights=weights):.4f}")
print(f"Rho:   {np.average(particles[:,2], weights=weights):.4f}")

In [ ]:
param_names = [r'$\beta$', r'$\gamma$', r'$\rho$']
colors = {
    'v1':  '#e74c3c',
    'v2':  '#e67e22',
    'v3':  '#2ecc71',
    'smc': '#3498db'
}
labels = {
    'v1':  'Rejection: 8 statistics',
    'v2':  'Rejection: inf+deg',
    'v3':  'Rejection: inf+deg+rewire',
    'smc': 'SMC-ABC'
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Marginal posterior distributions", fontsize=14)
 
for col in range(3):
    ax = axes[col]
    ax.hist(res_v1[:, col], bins=20, alpha=0.5, color=colors['v1'], label=labels['v1'], density=True)
    ax.hist(res_v2[:, col], bins=20, alpha=0.5, color=colors['v2'], label=labels['v2'], density=True)
    ax.hist(res_v3[:, col], bins=20, alpha=0.5, color=colors['v3'], label=labels['v3'], density=True)
    ax.hist(particles[:, col], bins=20, alpha=0.5, color=colors['smc'],
            weights=weights * len(weights), label=labels['smc'], density=True)
    ax.set_xlabel(param_names[col], fontsize=13)
    ax.set_ylabel("Densité")
    if col == 0:
        ax.legend(fontsize=8)
 
plt.tight_layout()
plt.savefig('posteriors_marginaux.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ==============================================================
# FIGURE : Marginal posteriors — 3 configurations (3x3 panel)
# ==============================================================

fig, axes = plt.subplots(3, 3, figsize=(12, 10))
fig.suptitle("Marginal posteriors — comparison of summary statistic configurations", 
             fontsize=13, y=1.02)

param_names = [r'$\beta$', r'$\gamma$', r'$\rho$']
configs = [
    (res_v1, '#e67e22', 'Config A: scalar summaries'),
    (res_v2, '#e74c3c', 'Config B: inf + deg'),
    (res_v3, '#2ecc71', 'Config C: inf + deg + rewire'),
]

xlims = {
    0: (0.05, 0.5),
    1: (0.02, 0.2),   
    2: (0.0,  0.8),  
}

for row, (res, color, label) in enumerate(configs):
    for col in range(3):
        ax = axes[row, col]
        ax.hist(res[:, col], bins=30, color=color, alpha=0.75, 
                density=True, edgecolor='white',
                range=xlims[col]) 
        ax.set_xlim(xlims[col])  
        
        mean = res[:, col].mean()
        ax.axvline(mean, color='black', linestyle='--', linewidth=1.5, label=f'μ={mean:.3f}')
        ax.legend(fontsize=8)
        
        if row == 0:
            ax.set_title(param_names[col], fontsize=12)
        if col == 0:
            ax.set_ylabel(label, fontsize=9, fontweight='bold')
        
        ax.set_xlabel(param_names[col], fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
n_steps = len(history)
fig, axes = plt.subplots(3, n_steps, figsize=(4 * n_steps, 10))
fig.suptitle("Evolution of the SMC-ABC posterior", fontsize=14)
 
for t, (epsilon, parts, w) in enumerate(history):
    for param_idx in range(3):
        ax = axes[param_idx, t]
        ax.hist(parts[:, param_idx], bins=20, weights=w * len(w),
                color='#3498db', alpha=0.7, density=True)
        ax.set_title(f'ε={epsilon}', fontsize=9)
        if t == 0:
            ax.set_ylabel(param_names[param_idx], fontsize=11)
        mean = np.average(parts[:, param_idx], weights=w)
        ax.axvline(mean, color='red', linestyle='--', linewidth=1.5, label=f'μ={mean:.3f}')
        if param_idx == 0:
            ax.legend(fontsize=7)
 
plt.tight_layout()
plt.savefig('smc_evolution.png', dpi=150, bbox_inches='tight')
plt.show()